<a href="https://colab.research.google.com/github/blublunalnal/afib_detection/blob/main/data_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!cp -r /content/drive/MyDrive/afib/db.zip /content/db.zip

In [1]:
!unzip  /content/drive/MyDrive/afib/db.zip -d /content/original_data/

Archive:  /content/drive/MyDrive/afib/db.zip
   creating: /content/original_data/db/
  inflating: /content/original_data/db/test.npz  
  inflating: /content/original_data/db/train.npz  
  inflating: /content/original_data/db/validate.npz  


In [3]:
!cp -r /content/drive/MyDrive/afib/relabeled_data.zip /content/relabeled_data.zip

In [2]:
!unzip  /content/drive/MyDrive/afib//relabeled_data.zip -d /content/relabeled_data/

Archive:  /content/drive/MyDrive/afib//relabeled_data.zip
   creating: /content/relabeled_data/relabeled_data/
  inflating: /content/relabeled_data/relabeled_data/db_vsm_combined_data.mat  
  inflating: /content/relabeled_data/relabeled_data/db_vsm_combined_label_q.mat  
  inflating: /content/relabeled_data/relabeled_data/db_vsm_combined_label_r.mat  
  inflating: /content/relabeled_data/relabeled_data/db_vsm_combined_sub_id.mat  
  inflating: /content/relabeled_data/relabeled_data/readme.rtf  


## functions

In [2]:
from numpy._core.multiarray import inner
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path
import pickle

def remove_nan_data(data_dict):
    """Remove samples containing NaN values"""
    no_nan_mask = ~np.isnan(data_dict['data']).any(axis=(1, 2))
    for k in data_dict.keys():
        data_dict[k] = data_dict[k][no_nan_mask]
    return data_dict

def load_original_data(data_path, file_name):
    data = np.load(Path(data_path) / file_name, allow_pickle=True)
    output = {}
    output['data'] = data['signal']
    output['qa_label'] = data['qa_label']
    output['rhythm'] = data['rhythm']

    params = pd.DataFrame(data['parameters'])
    params.rename(index=str, columns={0: 'timestamp', 1: 'stream', 2: 'ID'}, inplace=True)
    output['ID'] = np.array(params['ID'].to_list())
    output = remove_nan_data(output)
    return output

def load_relabeled_data(data_path):
    def load_from_mat(dir_path, file_name):
        file_mat = loadmat(Path(dir_path) / file_name)
        return file_mat.get(file_name[:-4])

    combined = {}
    combined['data'] = load_from_mat(data_path, 'db_vsm_combined_data.mat')
    combined['qa_label'] = load_from_mat(data_path, 'db_vsm_combined_label_q.mat')
    combined['rhythm'] = load_from_mat(data_path, 'db_vsm_combined_label_r.mat')
    combined['ID'] = load_from_mat(data_path, 'db_vsm_combined_sub_id.mat').flatten()
    combined['data'] = combined['data'].reshape(combined['data'].shape[0], combined['data'].shape[1], 1)

    num_classes_rhythm = 2
    num_classes_qa = 3
    combined['rhythm'] = np.eye(num_classes_rhythm)[combined['rhythm'].flatten().astype(int)]
    combined['qa_label'] = np.eye(num_classes_qa)[combined['qa_label'].flatten().astype(int)]

    relabeled_db = {}
    relabeled_vsm = {}
    db_mask = (combined['ID'] < 1000).flatten()
    vsm_mask = (combined['ID'] >= 1000).flatten()

    relabeled_db['data'] = combined['data'][db_mask, :]
    relabeled_db['qa_label'] = combined['qa_label'][db_mask, :]
    relabeled_db['rhythm'] = combined['rhythm'][db_mask, :]
    relabeled_db['ID'] = combined['ID'][db_mask].flatten()

    relabeled_vsm['data'] = combined['data'][vsm_mask, :]
    relabeled_vsm['qa_label'] = combined['qa_label'][vsm_mask, :]
    relabeled_vsm['rhythm'] = combined['rhythm'][vsm_mask, :]
    relabeled_vsm['ID'] = combined['ID'][vsm_mask].flatten()

    return combined, relabeled_db, relabeled_vsm


def as_void(arr):

    # Converts (N, 800, 1) -> (N, 800)
    if arr.ndim > 2:
        arr = arr.reshape(arr.shape[0], -1)
    # ensure contiguous memory (Crucial for .view())
    arr = np.ascontiguousarray(arr)

    return arr.view(np.dtype((np.void, arr.dtype.itemsize * arr.shape[1])))
    # arr.view() treats the array as a single block of byte, returns a numpy void scalar
    # to get the actual Python byte object, uses arr.view().item()

def update_labels_streamed_optimized(db_train, relabeled_db, chunk_size=100000):
    """
    Updates db_train in-place by matching signals against relabeled_db.
    Handles duplicates in db_train correctly.
    O(N) comparison by creating a lookup Hash Table by hashing each signal into a Python Byte Object.


    """
    print("Building hash map from relabeled data...")
    rel_void = as_void(relabeled_db['data']).flatten()

    # Map hash -> index in relabeled_db
    # Note: If relabeled_db has duplicates, the last one wins here.
    rel_map = {row.item(): i for i, row in enumerate(rel_void)}

    num_train = len(db_train['data'])
    total_updated = 0

    print(f"Streaming {num_train} rows and updating in-place...")

    for start_idx in range(0, num_train, chunk_size):
        end_idx = min(start_idx + chunk_size, num_train)
        train_chunk_void = as_void(db_train['data'][start_idx:end_idx]).flatten()

        for i, row in enumerate(train_chunk_void):
            row_bytes = row.item()

            if row_bytes in rel_map:
                rel_idx = rel_map[row_bytes]
                train_idx = start_idx + i

                # Perform the update IMMEDIATELY
                db_train['rhythm'][train_idx] = relabeled_db['rhythm'][rel_idx]
                db_train['qa_label'][train_idx] = relabeled_db['qa_label'][rel_idx]

                total_updated += 1

        if end_idx % (chunk_size * 5) == 0 or end_idx == num_train:
            print(f"Processed {end_idx}/{num_train} rows...")

    print(f"Success: Updated {total_updated} instances in db_train.")
    return db_train



def check_duplicates(data_dict, epsilon=1e-10):
    """
    Identifies duplicate signals, verifies they are numerically identical,
    and checks for label consistency.
    """
    print("Hashing signals to find potential duplicates...")
    data_void = as_void(data_dict['data']).flatten()

    # 1. Find indices of all duplicates
    unique, counts = np.unique(data_void, return_counts=True)
    duplicate_blobs = unique[counts > 1]

    if len(duplicate_blobs) == 0:
        print("No duplicate signals found based on hash.")
        return []

    results = []

    print(f"Verifying {len(duplicate_blobs)} groups of duplicates...")

    for blob in duplicate_blobs:
        # Get all indices where this specific signal appears
        indices = np.where(data_void == blob)[0]

        # 2. Euclidean Distance Check
        # Compare all signals in the group to the first one in the group
        base_signal = data_dict['data'][indices[0]].flatten().astype(np.float64)
        is_numerically_identical = True

        for idx in indices[1:]:
            comp_signal = data_dict['data'][idx].flatten().astype(np.float64)
            dist = np.sqrt(np.sum((base_signal - comp_signal)**2))

            if dist > epsilon:
                is_numerically_identical = False
                break

        # 3. Label Consistency Check
        # Check if all instances have the same rhythm and QA labels
        rhythms = data_dict['rhythm'][indices]
        qa_labels = data_dict['qa_label'][indices]

        # np.unique with axis=0 finds unique rows (labels)
        unique_rhythms = np.unique(rhythms, axis=0)
        unique_qas = np.unique(qa_labels, axis=0)

        rythm_consistent = len(unique_rhythms) == 1
        qua_consistent = len(unique_qas) == 1


        results.append({
            'indices': indices,
            'count': len(indices),
            'numerically_exact': is_numerically_identical,
            'rythm consistent': rythm_consistent,
            'qa consistent': qua_consistent,

        })

    # # Summary Printout
    # for res in results:
    #     status = "PASS" if (res['numerically_exact'] and res['rythm consistent'] and res['qa consistent']) else "FAIL"
    #     print(f"[{status}] Group at indices {res['indices']}: "
    #           f"Distance Match: {res['numerically_exact']}, "
    #           f"Rythm Consistent: {res['rythm consistent']}",
    #           f"QA consistent: {res['qa consistent']}")

    return results


def drop_duplicates(data_dict, duplicate_results):
    """
    Drops duplicate signals based on the results from check_duplicates:
    1. If signal and labels are identical: keep one, drop the rest
    2. If signal is duplicate but labels differ: drop all instances

    Parameters:
    -----------
    data_dict : dict
        Dictionary containing 'data', 'rhythm', 'qa_label', and 'ID' arrays
    duplicate_results : list
        Output from check_duplicates() function

    Returns:
    --------
    dict : New data_dict with duplicates removed
    int : Number of samples removed
    """
    if len(duplicate_results) == 0:
        print("No duplicates to remove.")
        return data_dict, 0

    indices_to_drop = set()

    print(f"Processing {len(duplicate_results)} groups of duplicates...")

    for res in duplicate_results:
        indices = res['indices']

        # Check if this is a valid duplicate group (numerically identical)
        if not res['numerically_exact']:
            print(f"  Skipping indices {indices} - not numerically identical (hash collision)")
            continue

        # Check if labels are consistent
        labels_consistent = res['rythm consistent'] and res['qa consistent']

        if labels_consistent:
            # Case 1: Signal and labels identical - keep first, drop rest
            indices_to_drop.update(indices[1:])
            #print(f"  Keeping 1/{len(indices)} identical samples at indices {indices}")
        else:
            # Case 2: Signal identical but labels differ - drop ALL
            indices_to_drop.update(indices)
            #print(f"  Dropping all {len(indices)} samples with conflicting labels at indices {indices}")


    total_samples = len(data_dict['data'])
    keep_mask = np.ones(total_samples, dtype=bool)
    keep_mask[list(indices_to_drop)] = False

    # Create new data_dict with filtered data
    filtered_dict = {
        'data': data_dict['data'][keep_mask],
        'rhythm': data_dict['rhythm'][keep_mask],
        'qa_label': data_dict['qa_label'][keep_mask],
        'ID': data_dict['ID'][keep_mask]
    }

    num_dropped = len(indices_to_drop)
    print(f"\nSummary:")
    print(f"  Original samples: {total_samples}")
    print(f"  Dropped samples: {num_dropped}")
    print(f"  Remaining samples: {total_samples - num_dropped}")

    return filtered_dict, num_dropped


def check_duplicates_in_chunks(data_dict, epsilon=1e-10, chunk_size=100000):
    """
    Identifies duplicate signals across all chunks by building a global hash map.

    Parameters:
    -----------
    data_dict : dict
        Dictionary containing 'data', 'rhythm', 'qa_label', and 'ID' arrays
    epsilon : float
        Tolerance for numerical comparison
    chunk_size : int
        Number of samples to process at once
    """
    total_samples = len(data_dict['data'])
    print(f"Processing {total_samples} samples in chunks of {chunk_size}...")

    # Dictionary to track hash -> list of GLOBAL indices
    hash_to_indices = {}

    # Step 1: Build global hash map by processing chunks
    num_chunks = (total_samples + chunk_size - 1) // chunk_size

    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min(start_idx + chunk_size, total_samples)

        print(f"  Chunk {chunk_idx + 1}/{num_chunks}: Hashing samples {start_idx} to {end_idx-1}")

        # Get chunk and hash it
        chunk_data = data_dict['data'][start_idx:end_idx]
        chunk_void = as_void(chunk_data).flatten()

        # Map each hash to its GLOBAL indices
        for local_idx, hash_val in enumerate(chunk_void):
            global_idx = start_idx + local_idx
            hash_bytes = hash_val.tobytes()  # Convert to bytes for dict key

            if hash_bytes not in hash_to_indices:
                hash_to_indices[hash_bytes] = []
            hash_to_indices[hash_bytes].append(global_idx)

        # Clear chunk from memory
        del chunk_data, chunk_void

    # Step 2: Find duplicate groups (hashes that appear more than once)
    duplicate_groups = {k: v for k, v in hash_to_indices.items() if len(v) > 1}

    # Clear hash map to free memory
    del hash_to_indices

    if len(duplicate_groups) == 0:
        print("No duplicate signals found based on hash.")
        return []

    print(f"\nFound {len(duplicate_groups)} groups of potential duplicates")
    print(f"Total duplicate instances: {sum(len(v) for v in duplicate_groups.values())}")
    print(f"Verifying duplicates...")

    results = []

    # Step 3: Verify each duplicate group using GLOBAL indices
    for group_idx, (hash_val, indices) in enumerate(duplicate_groups.items(), 1):
        if group_idx % 100 == 0:
            print(f"  Verified {group_idx}/{len(duplicate_groups)} groups...")

        indices = np.array(indices)

        # Euclidean Distance Check using global indices
        base_signal = data_dict['data'][indices[0]].flatten().astype(np.float64)
        is_numerically_identical = True

        for idx in indices[1:]:
            comp_signal = data_dict['data'][idx].flatten().astype(np.float64)
            dist = np.sqrt(np.sum((base_signal - comp_signal)**2))

            if dist > epsilon:
                is_numerically_identical = False
                break

        # Label Consistency Check using global indices
        rhythms = data_dict['rhythm'][indices]
        qa_labels = data_dict['qa_label'][indices]

        unique_rhythms = np.unique(rhythms, axis=0)
        unique_qas = np.unique(qa_labels, axis=0)

        rhythm_consistent = len(unique_rhythms) == 1
        qa_consistent = len(unique_qas) == 1

        results.append({
            'indices': indices,
            'count': len(indices),
            'numerically_exact': is_numerically_identical,
            'rythm consistent': rhythm_consistent,
            'qa consistent': qa_consistent,
        })

    print(f"\nCompleted verification of {len(results)} duplicate groups")

    # Summary statistics
    total_duplicates = sum(res['count'] for res in results)
    exact_matches = sum(1 for res in results if res['numerically_exact'])
    label_consistent = sum(1 for res in results if res['rythm consistent'] and res['qa consistent'])

    print(f"\nSummary:")
    print(f"  Total duplicate instances: {total_duplicates}")
    print(f"  Numerically exact groups: {exact_matches}/{len(results)}")
    print(f"  Label consistent groups: {label_consistent}/{len(results)}")

    return results



def save_as_pickle(data, file_path):
  with open(file_path, 'wb') as f:
      pickle.dump(data, f)
  print(f"Data saved to {file_path}")





## data loading

In [3]:
#ori_train = load_original_data('/content/original_data/db/', 'train.npz')
ori_val = load_original_data('/content/original_data/db/', 'validate.npz')
ori_test = load_original_data('/content/original_data/db/', 'test.npz')
#_, relabeled_db, relabeled_vsm = load_relabeled_data('/content/relabeled_data/relabeled_data/')

## check data integrity

### check relabeled deepbeat signals

In [14]:
relabeled_db_results = check_duplicates(relabeled_db, epsilon=1e-10)

Hashing signals to find potential duplicates...
Verifying 401 groups of duplicates...
[FAIL] Group at indices [62062 64723]: Distance Match: True, Rythm Consistent: True QA consistent: False
[PASS] Group at indices [ 9819 54129]: Distance Match: True, Rythm Consistent: True QA consistent: True
[PASS] Group at indices [60204 60505]: Distance Match: True, Rythm Consistent: True QA consistent: True
[PASS] Group at indices [69190 70705]: Distance Match: True, Rythm Consistent: True QA consistent: True
[FAIL] Group at indices [62060 64721]: Distance Match: True, Rythm Consistent: True QA consistent: False
[FAIL] Group at indices [63408 64350]: Distance Match: True, Rythm Consistent: True QA consistent: False
[PASS] Group at indices [61648 61907]: Distance Match: True, Rythm Consistent: True QA consistent: True
[PASS] Group at indices [61632 61891]: Distance Match: True, Rythm Consistent: True QA consistent: True
[PASS] Group at indices [61695 61934]: Distance Match: True, Rythm Consistent: 

In [34]:
relabeled_db_r_df = pd.DataFrame(relabeled_db_results)
print(f"# duplicates with mismatch signal (hash collision case): {len(relabeled_db_r_df[relabeled_db_r_df['numerically_exact'] == False])}")
print(f"# duplicates with mismatch rythm label: {len(relabeled_db_r_df[relabeled_db_r_df['rythm consistent'] == False])}")
print(f"# duplicates with mismatch Qa label: {np.sum((relabeled_db_r_df[relabeled_db_r_df['qa consistent'] == False]['count']))}")

# duplicates with mismatch signal (hash collision case): 0
# duplicates with mismatch rythm label: 0
# duplicates with mismatch Qa label: 140


In [37]:
print(relabeled_db_r_df.head(2))
 # printing an example to make sure it's not inconsistent due to rounding error
print(relabeled_db['qa_label'][62062])
print(relabeled_db['qa_label'][64723])

          indices  count  numerically_exact  rythm consistent  qa consistent
0  [62062, 64723]      2               True              True          False
1   [9819, 54129]      2               True              True           True
[0. 1. 0.]
[0. 0. 1.]


### clean relabeled deepbeat
Found some data with duplicate signals but with different QA label. The following drops duplicate signals based on the results from check_duplicates:
1. If signal and labels are identical: keep one, drop the rest
2. If signal is duplicate but labels differ: drop all instances


In [27]:
cleaned_relabeled_db, _ = drop_duplicates(relabeled_db, relabeled_db_results )

Processing 401 groups of duplicates...

Summary:
  Original samples: 73147
  Dropped samples: 471
  Remaining samples: 72676


### check new vsm signals

No need to clean as there is no duplicate

In [38]:
relabeled_vsm_results = check_duplicates(relabeled_vsm, epsilon=1e-10)

Hashing signals to find potential duplicates...
No duplicate signals found based on hash.


### check original test data (deepbeat)
No need to clean as there is no duplicate

In [40]:
test_dup_results = check_duplicates(ori_test, epsilon=1e-10)

Hashing signals to find potential duplicates...
No duplicate signals found based on hash.


### check original validation data (deepbeat)
No need to clean as there is no duplicate

In [41]:
val_dup_results = check_duplicates(ori_val, epsilon=1e-10)

Hashing signals to find potential duplicates...
No duplicate signals found based on hash.


### check original training data (deepbeat)
No need to clean as there is no duplicate

In [4]:
train_dup_results = check_duplicates_in_chunks(ori_train, epsilon=1e-10, chunk_size=1000)

Processing 2799784 samples in chunks of 1000...
  Chunk 1/2800: Hashing samples 0 to 999
  Chunk 2/2800: Hashing samples 1000 to 1999
  Chunk 3/2800: Hashing samples 2000 to 2999
  Chunk 4/2800: Hashing samples 3000 to 3999
  Chunk 5/2800: Hashing samples 4000 to 4999
  Chunk 6/2800: Hashing samples 5000 to 5999
  Chunk 7/2800: Hashing samples 6000 to 6999
  Chunk 8/2800: Hashing samples 7000 to 7999
  Chunk 9/2800: Hashing samples 8000 to 8999
  Chunk 10/2800: Hashing samples 9000 to 9999
  Chunk 11/2800: Hashing samples 10000 to 10999
  Chunk 12/2800: Hashing samples 11000 to 11999
  Chunk 13/2800: Hashing samples 12000 to 12999
  Chunk 14/2800: Hashing samples 13000 to 13999
  Chunk 15/2800: Hashing samples 14000 to 14999
  Chunk 16/2800: Hashing samples 15000 to 15999
  Chunk 17/2800: Hashing samples 16000 to 16999
  Chunk 18/2800: Hashing samples 17000 to 17999
  Chunk 19/2800: Hashing samples 18000 to 18999
  Chunk 20/2800: Hashing samples 19000 to 19999
  Chunk 21/2800: Hashing 

In [5]:
train_dup_df = pd.DataFrame(train_dup_results)
print(f"# duplicates with mismatch signal (hash collision case): {len(train_dup_df[train_dup_df['numerically_exact'] == False])}")
print(f"# duplicates with mismatch rythm label: {np.sum(train_dup_df[train_dup_df['rythm consistent'] == False]['count'])}")
print(f"# duplicates with mismatch Qa label: {np.sum((train_dup_df[train_dup_df['qa consistent'] == False]['count']))}")
print(f"Total groups of duplicates: {len(train_dup_df)}")

# duplicates with mismatch signal (hash collision case): 0
# duplicates with mismatch rythm label: 125
# duplicates with mismatch Qa label: 0
Total groups of duplicates: 561


### clean original training data

In [5]:
ori_train_clean, num_dropped = drop_duplicates(ori_train, train_dup_results)

Processing 561 groups of duplicates...

Summary:
  Original samples: 2799784
  Dropped samples: 3685
  Remaining samples: 2796099


In [8]:
save_as_pickle(ori_train_clean, '/content/drive/MyDrive/afib/data/ori_train_clean.pkl')

Data saved to /content/drive/MyDrive/afib/data/ori_train_clean.pkl


### update (cleaned) original training data with (cleaned) relabeled data

- function showed that it updated 72676 instances in (cleaned) original training data.
- as len (cleaned_relabeled_db['data']) = 72676, all the relabeled data are in the training data.

In [22]:
ori_train_clean_updated = update_labels_streamed_optimized(ori_train_clean, cleaned_relabeled_db, chunk_size=1000)

Building hash map from relabeled data...
Streaming 2796099 rows and updating in-place...
Processed 5000/2796099 rows...
Processed 10000/2796099 rows...
Processed 15000/2796099 rows...
Processed 20000/2796099 rows...
Processed 25000/2796099 rows...
Processed 30000/2796099 rows...
Processed 35000/2796099 rows...
Processed 40000/2796099 rows...
Processed 45000/2796099 rows...
Processed 50000/2796099 rows...
Processed 55000/2796099 rows...
Processed 60000/2796099 rows...
Processed 65000/2796099 rows...
Processed 70000/2796099 rows...
Processed 75000/2796099 rows...
Processed 80000/2796099 rows...
Processed 85000/2796099 rows...
Processed 90000/2796099 rows...
Processed 95000/2796099 rows...
Processed 100000/2796099 rows...
Processed 105000/2796099 rows...
Processed 110000/2796099 rows...
Processed 115000/2796099 rows...
Processed 120000/2796099 rows...
Processed 125000/2796099 rows...
Processed 130000/2796099 rows...
Processed 135000/2796099 rows...
Processed 140000/2796099 rows...
Process

In [24]:
save_as_pickle(ori_train_clean_updated, '/content/drive/MyDrive/afib/data/ori_train_clean_updated.pkl')

Data saved to /content/drive/MyDrive/afib/data/ori_train_clean_updated.pkl


## Check Data Distribution

In [5]:
print("cleaned, updated train data(rhythm label)")
counts = np.sum(ori_train_clean_updated['rhythm'], axis=0)
print(f"Class 0 Total: {counts[0]}")
print(f"Class 1 Total: {counts[1]}")
print(f"Total Samples: {len(ori_train_clean_updated['rhythm'])}")
print(f"positive to negative ratio: {counts[1] / counts[0]}")
print("="*60)
print("validation data (rhythm)")
counts = np.sum(ori_val['rhythm'], axis=0)
print(f"Class 0 Total: {counts[0]}")
print(f"Class 1 Total: {counts[1]}")
print(f"Total Samples: {len(ori_val['rhythm'])}")
print(f"positive to negative ratio: {counts[1] / counts[0]}")
print("="*60)
print("test data(rhythm)")
counts = np.sum(ori_test['rhythm'], axis=0)
print(f"Class 0 Total: {counts[0]}")
print(f"Class 1 Total: {counts[1]}")
print(f"Total Samples: {len(ori_test['rhythm'])}")
print(f"positive to negative ratio: {counts[1] / counts[0]}")

cleaned, updated train data(rhythm label)
Class 0 Total: 1527399.0
Class 1 Total: 1268700.0
Total Samples: 2796099
positive to negative ratio: 0.830627753455384
validation data (rhythm)
Class 0 Total: 471175.0
Class 1 Total: 47607.0
Total Samples: 518782
positive to negative ratio: 0.10103889554738998
test data(rhythm)
Class 0 Total: 13387.0
Class 1 Total: 4230.0
Total Samples: 17617
positive to negative ratio: 0.31597819924354553
